In [6]:
# Next Ascent Hillclimbing using 1-bit Hamming Distance neighbours
import random
import time

dimacs = 'uf20-01.cnf'
with open(dimacs, 'r') as file:
    dimacs_content = file.readlines()

clauses = []
num_vars = None
num_clauses = None
for line in dimacs_content:
    line = line.strip()
    if line.startswith('c') or line == '' or line.startswith('%') or line.startswith('0'):
        continue
    if line.startswith('p'):
        _, _, num_vars, num_clauses = line.split()
        num_vars = int(num_vars)
        num_clauses = int(num_clauses)
        continue
    clause = list(map(int, line.split()[:-1]))
    clauses.append(clause)

assert len(clauses) == num_clauses, f"Expected {num_clauses} clauses but got {len(clauses)}"

def evaluate_fitness(solution):
    satisfied_clauses = 0
    for clause in clauses:
        if any((literal > 0 and solution[abs(literal) - 1]) or (literal < 0 and not solution[abs(literal) - 1]) for literal in clause):
            satisfied_clauses += 1
    return satisfied_clauses


def generate_neighbours(solution):
    neighbours = []
    for i in range(len(solution)):
        neighbour = list(solution)
        neighbour[i] = 1 - neighbour[i]  
        neighbours.append(neighbour)
    return neighbours


def next_ascent_hillclimbing(initial_solution, max_iterations):
    current_solution = initial_solution
    best_solution = current_solution
    best_fitness = evaluate_fitness(current_solution)

    for _ in range(max_iterations):
        neighbours = generate_neighbours(current_solution)
        random.shuffle(neighbours)  
        for neighbour in neighbours:
            fitness = evaluate_fitness(neighbour)
            if fitness > best_fitness:
                best_solution = neighbour
                best_fitness = fitness
        current_solution = best_solution

    return best_solution


num_runs = 30
max_iterations = 10000


times = []
best_solutions = []
best_fitness_values = []


for run in range(num_runs):

    initial_solution = [random.choice([0, 1]) for _ in range(num_vars)]
    

    start_time = time.time()
    

    best_solution = next_ascent_hillclimbing(initial_solution, max_iterations)
    
    end_time = time.time()
    
    time_taken = end_time - start_time
    times.append(time_taken)
    
    best_fitness = evaluate_fitness(best_solution)
    best_solutions.append(best_solution)
    best_fitness_values.append(best_fitness)
    
    print(f"Run {run + 1}: Best solution satisfies {best_fitness}/{num_clauses} clauses, Time taken: {time_taken:.4f} seconds")


print("\nSummary of 30 runs:")
print(f"Average time: {sum(times) / num_runs:.4f} seconds")
print(f"Maximum clauses satisfied in any run: {max(best_fitness_values)}")
print(f"Average clauses satisfied: {sum(best_fitness_values) / num_runs}")


Run 1: Best solution satisfies 87/91 clauses, Time taken: 8.4645 seconds
Run 2: Best solution satisfies 88/91 clauses, Time taken: 11.2049 seconds
Run 3: Best solution satisfies 87/91 clauses, Time taken: 11.5387 seconds
Run 4: Best solution satisfies 90/91 clauses, Time taken: 11.9364 seconds
Run 5: Best solution satisfies 89/91 clauses, Time taken: 12.6673 seconds
Run 6: Best solution satisfies 88/91 clauses, Time taken: 10.4490 seconds
Run 7: Best solution satisfies 89/91 clauses, Time taken: 11.4278 seconds
Run 8: Best solution satisfies 86/91 clauses, Time taken: 10.3113 seconds
Run 9: Best solution satisfies 88/91 clauses, Time taken: 11.1605 seconds
Run 10: Best solution satisfies 88/91 clauses, Time taken: 9.8628 seconds
Run 11: Best solution satisfies 90/91 clauses, Time taken: 11.8795 seconds
Run 12: Best solution satisfies 88/91 clauses, Time taken: 9.6170 seconds
Run 13: Best solution satisfies 88/91 clauses, Time taken: 10.3731 seconds
Run 14: Best solution satisfies 89/91

In [7]:
# 2) Multistart Next Ascent Hillclimbing (MSNAHC):
import random
import time


def read_dimacs(file):
    clauses = []
    num_vars = None
    num_clauses = None
    with open(file, 'r') as file:
        dimacs_content = file.readlines()
    for line in dimacs_content:
        line = line.strip()
        if line.startswith('c') or line == '' or line.startswith('%') or line.startswith('0'):
            continue
        if line.startswith('p'):
            _, _, num_vars, num_clauses = line.split()
            num_vars = int(num_vars)
            num_clauses = int(num_clauses)
            continue
        clause = list(map(int, line.split()[:-1]))
        clauses.append(clause)
    assert len(clauses) == num_clauses, f"Expected {num_clauses} clauses but got {len(clauses)}"
    return clauses, num_vars, num_clauses


def evaluate_fitness(solution, clauses):
    satisfied_clauses = 0
    for clause in clauses:
        if any((literal > 0 and solution[abs(literal) - 1]) or (literal < 0 and not solution[abs(literal) - 1]) for literal in clause):
            satisfied_clauses += 1
    return satisfied_clauses


def generate_neighbours(solution):
    neighbours = []
    for i in range(len(solution)):
        neighbour = list(solution)
        neighbour[i] = 1 - neighbour[i]  
        neighbours.append(neighbour)
    return neighbours


def msnahc(clauses, num_vars, max_evaluations=20000000):
    global_evaluations = 0
    best_solution = None
    best_fitness = -1
    best_eval_count = 0

    while global_evaluations < max_evaluations:

        current_solution = [random.choice([0, 1]) for _ in range(num_vars)]
        current_fitness = evaluate_fitness(current_solution, clauses)
        eval_count = 1

        while eval_count + global_evaluations < max_evaluations:
            neighbours = generate_neighbours(current_solution)
            random.shuffle(neighbours)  
            local_best = False

            for neighbour in neighbours:
                neighbour_fitness = evaluate_fitness(neighbour, clauses)
                eval_count += 1

                if neighbour_fitness > current_fitness:
                    current_solution = neighbour
                    current_fitness = neighbour_fitness
                    local_best = True
                    break
            

            if not local_best:
                break

        global_evaluations += eval_count


        if current_fitness > best_fitness:
            best_solution = current_solution
            best_fitness = current_fitness
            best_eval_count = global_evaluations


        if best_fitness == len(clauses):
            break

    return best_solution, best_fitness, best_eval_count, global_evaluations


dimacs_file = 'uf20-01.cnf'
clauses, num_vars, num_clauses = read_dimacs(dimacs_file)

num_runs = 30
max_iterations = 10000
max_evaluations = 20000000


times = []
best_solutions = []
best_fitness_values = []
eval_counts = []

for run in range(num_runs):
    start_time = time.time()
    
    best_solution, best_fitness, eval_count, global_evaluations = msnahc(clauses, num_vars, max_evaluations)
    
    end_time = time.time()
    time_taken = end_time - start_time
    

    times.append(time_taken)
    best_solutions.append(best_solution)
    best_fitness_values.append(best_fitness)
    eval_counts.append(eval_count)
    
    print(f"Run {run + 1}: Best fitness = {best_fitness}, Evaluations = {eval_count}, Time = {time_taken:.4f} seconds")


print("\nSummary of MSNAHC after 30 runs:")
print(f"Average time: {sum(times) / num_runs:.4f} seconds")
print(f"Average evaluations: {sum(eval_counts) / num_runs}")
print(f"Maximum clauses satisfied in any run: {max(best_fitness_values)}")
print(f"Average clauses satisfied: {sum(best_fitness_values) / num_runs}")


Run 1: Best fitness = 91, Evaluations = 610, Time = 0.0571 seconds
Run 2: Best fitness = 91, Evaluations = 1449, Time = 0.1457 seconds
Run 3: Best fitness = 91, Evaluations = 452, Time = 0.0395 seconds
Run 4: Best fitness = 91, Evaluations = 830, Time = 0.0690 seconds
Run 5: Best fitness = 91, Evaluations = 517, Time = 0.0425 seconds
Run 6: Best fitness = 91, Evaluations = 939, Time = 0.0671 seconds
Run 7: Best fitness = 91, Evaluations = 1691, Time = 0.0977 seconds
Run 8: Best fitness = 91, Evaluations = 305, Time = 0.0257 seconds
Run 9: Best fitness = 91, Evaluations = 248, Time = 0.0201 seconds
Run 10: Best fitness = 91, Evaluations = 151, Time = 0.0120 seconds
Run 11: Best fitness = 91, Evaluations = 45, Time = 0.0030 seconds
Run 12: Best fitness = 91, Evaluations = 79, Time = 0.0070 seconds
Run 13: Best fitness = 91, Evaluations = 3125, Time = 0.2228 seconds
Run 14: Best fitness = 91, Evaluations = 1944, Time = 0.1671 seconds
Run 15: Best fitness = 91, Evaluations = 1028, Time = 0

In [8]:
# Variable Neighbourhood Ascent (VNA)
import random
import time
import itertools


def read_dimacs(file):
    clauses = []
    num_vars = None
    num_clauses = None
    with open(file, 'r') as file:
        dimacs_content = file.readlines()
    for line in dimacs_content:
        line = line.strip()
        if line.startswith('c') or line == '' or line.startswith('%') or line.startswith('0'):
            continue
        if line.startswith('p'):
            _, _, num_vars, num_clauses = line.split()
            num_vars = int(num_vars)
            num_clauses = int(num_clauses)
            continue
        clause = list(map(int, line.split()[:-1]))
        clauses.append(clause)
    assert len(clauses) == num_clauses, f"Expected {num_clauses} clauses but got {len(clauses)}"
    return clauses, num_vars, num_clauses


def evaluate_fitness(solution, clauses):
    satisfied_clauses = 0
    for clause in clauses:
        if any((literal > 0 and solution[abs(literal) - 1]) or (literal < 0 and not solution[abs(literal) - 1]) for literal in clause):
            satisfied_clauses += 1
    return satisfied_clauses


def generate_k_bit_neighbours(solution, k):
    neighbours = []
    indices = list(range(len(solution)))
    for bit_indices in itertools.combinations(indices, k):
        neighbour = list(solution)
        for i in bit_indices:
            neighbour[i] = 1 - neighbour[i]
        neighbours.append(neighbour)
    return neighbours


def variable_neighbourhood_ascent(initial_solution, clauses, max_iterations):
    current_solution = initial_solution
    best_solution = current_solution
    # Fitness of the current solution
    best_fitness = evaluate_fitness(current_solution, clauses)

    k = 1
    while k <= 3:  # Explore 1-bit, 2-bit, and 3-bit Hamming neighbourhoods
        neighbours = generate_k_bit_neighbours(current_solution, k)
        random.shuffle(neighbours)

        improvement_found = False
        for neighbour in neighbours:
            # Evaluate the fitness of the neighbour
            fitness = evaluate_fitness(neighbour, clauses)
            if fitness > best_fitness:  # Improvement found
                best_solution = neighbour
                best_fitness = fitness
                improvement_found = True
                break

        if improvement_found:
            current_solution = best_solution
            k = 1  # Restart with 1-bit neighbours
        else:
            k += 1  # Move to the next larger neighbourhood

    return best_solution


dimacs_file = 'uf20-01.cnf'
clauses, num_vars, num_clauses = read_dimacs(dimacs_file)

# Execution for 30 runs
num_runs = 30
max_iterations = 10000

times = []
best_solutions = []
best_fitness_values = []

# Perform the algorithm for 30 independent runs
for run in range(num_runs):
    # Generate a random initial solution
    initial_solution = [random.choice([0, 1]) for _ in range(num_vars)]

    # Measure the time for each run
    start_time = time.time()

    # Run the Variable Neighbourhood Ascent algorithm
    best_solution = variable_neighbourhood_ascent(initial_solution, clauses, max_iterations)

    end_time = time.time()
    time_taken = end_time - start_time
    times.append(time_taken)

    # Fitness of the best solution found
    best_fitness = evaluate_fitness(best_solution, clauses)
    best_solutions.append(best_solution)
    best_fitness_values.append(best_fitness)

    print(f"Run {run + 1}: Best solution satisfies {best_fitness}/{num_clauses} clauses, Time taken: {time_taken:.4f} seconds")

# Summary of results after 30 runs
print("\nSummary of 30 runs:")
print(f"Average time: {sum(times) / num_runs:.4f} seconds")
print(f"Maximum clauses satisfied in any run: {max(best_fitness_values)}")
print(f"Average clauses satisfied: {sum(best_fitness_values) / num_runs}")

Run 1: Best solution satisfies 89/91 clauses, Time taken: 0.1140 seconds
Run 2: Best solution satisfies 88/91 clauses, Time taken: 0.0885 seconds
Run 3: Best solution satisfies 88/91 clauses, Time taken: 0.0880 seconds
Run 4: Best solution satisfies 89/91 clauses, Time taken: 0.1227 seconds
Run 5: Best solution satisfies 91/91 clauses, Time taken: 0.1988 seconds
Run 6: Best solution satisfies 90/91 clauses, Time taken: 0.1108 seconds
Run 7: Best solution satisfies 89/91 clauses, Time taken: 0.1208 seconds
Run 8: Best solution satisfies 88/91 clauses, Time taken: 0.0967 seconds
Run 9: Best solution satisfies 89/91 clauses, Time taken: 0.1404 seconds
Run 10: Best solution satisfies 90/91 clauses, Time taken: 0.1194 seconds
Run 11: Best solution satisfies 89/91 clauses, Time taken: 0.1049 seconds
Run 12: Best solution satisfies 91/91 clauses, Time taken: 0.1876 seconds
Run 13: Best solution satisfies 91/91 clauses, Time taken: 0.1505 seconds
Run 14: Best solution satisfies 91/91 clauses, 

In [9]:
# Multi-Start Variable Neighbourhood Ascent (MSVNA)
import random
import itertools
import time


def read_dimacs(file):
    clauses = []
    num_vars = None
    num_clauses = None
    with open(file, 'r') as file:
        dimacs_content = file.readlines()
    for line in dimacs_content:
        line = line.strip()
        if line.startswith('c') or line == '' or line.startswith('%') or line.startswith('0'):
            continue
        if line.startswith('p'):
            _, _, num_vars, num_clauses = line.split()
            num_vars = int(num_vars)
            num_clauses = int(num_clauses)
            continue
        clause = list(map(int, line.split()[:-1]))
        clauses.append(clause)
    assert len(clauses) == num_clauses, f"Expected {num_clauses} clauses but got {len(clauses)}"
    return clauses, num_vars, num_clauses


def evaluate_fitness(solution, clauses):
    satisfied_clauses = 0
    for clause in clauses:
        if any((literal > 0 and solution[abs(literal) - 1]) or (literal < 0 and not solution[abs(literal) - 1]) for literal in clause):
            satisfied_clauses += 1
    return satisfied_clauses


def multi_start_vna(max_iterations, max_evaluations):
    total_evaluations = 0
    best_global_solution = None
    best_global_fitness = 0

    while total_evaluations < max_evaluations:
        initial_solution = [random.choice([0, 1]) for _ in range(num_vars)]
        current_solution = initial_solution
        best_solution = current_solution
        best_fitness = evaluate_fitness(current_solution,clauses)

        evaluations = 0
        k = 1  
        while k <= 3 and evaluations < max_iterations:
            neighbours = generate_k_bit_neighbours(current_solution, k)
            random.shuffle(neighbours)
            
            improvement_found = False
            for neighbour in neighbours:
                fitness = evaluate_fitness(neighbour,clauses)
                evaluations += 1
                total_evaluations += 1

                if fitness > best_fitness:
                    best_solution = neighbour
                    best_fitness = fitness
                    improvement_found = True
                    break

                if total_evaluations >= max_evaluations:
                    break

            if improvement_found:
                current_solution = best_solution
                k = 1
            else:
                k += 1

            if total_evaluations >= max_evaluations:
                break

        if best_fitness > best_global_fitness:
            best_global_solution = best_solution
            best_global_fitness = best_fitness

        if best_global_fitness == num_clauses:  
            break

    return best_global_solution, total_evaluations


dimacs_file = 'uf20-01.cnf'
clauses, num_vars, num_clauses = read_dimacs(dimacs_file)

num_runs = 30
max_iterations = 10000
max_evaluations = 10_000_000  

times = []
best_solutions = []
best_fitness_values = []
evaluations_list = []

for run in range(num_runs):
    start_time = time.time()

    best_solution, total_evaluations = multi_start_vna(max_iterations, max_evaluations)
    
    end_time = time.time()
    
    time_taken = end_time - start_time
    times.append(time_taken)
    
    best_fitness = evaluate_fitness(best_solution,clauses)
    best_solutions.append(best_solution)
    best_fitness_values.append(best_fitness)
    evaluations_list.append(total_evaluations)
    
    print(f"Run {run + 1}: Best solution satisfies {best_fitness}/{num_clauses} clauses, Time taken: {time_taken:.4f} seconds, Total evaluations: {total_evaluations}")

print("\nSummary of 30 runs:")
print(f"Average time: {sum(times) / num_runs:.4f} seconds")
print(f"Maximum clauses satisfied in any run: {max(best_fitness_values)}")
print(f"Average clauses satisfied: {sum(best_fitness_values) / num_runs}")
print(f"Average evaluations per run: {sum(evaluations_list) / num_runs}")

Run 1: Best solution satisfies 91/91 clauses, Time taken: 0.1886 seconds, Total evaluations: 2428
Run 2: Best solution satisfies 91/91 clauses, Time taken: 0.7372 seconds, Total evaluations: 8402
Run 3: Best solution satisfies 91/91 clauses, Time taken: 0.2454 seconds, Total evaluations: 2735
Run 4: Best solution satisfies 91/91 clauses, Time taken: 0.3087 seconds, Total evaluations: 4280
Run 5: Best solution satisfies 91/91 clauses, Time taken: 0.4643 seconds, Total evaluations: 6403
Run 6: Best solution satisfies 91/91 clauses, Time taken: 0.1155 seconds, Total evaluations: 1555
Run 7: Best solution satisfies 91/91 clauses, Time taken: 0.0945 seconds, Total evaluations: 1431
Run 8: Best solution satisfies 91/91 clauses, Time taken: 0.0884 seconds, Total evaluations: 1445
Run 9: Best solution satisfies 91/91 clauses, Time taken: 0.0989 seconds, Total evaluations: 1378
Run 10: Best solution satisfies 91/91 clauses, Time taken: 0.1145 seconds, Total evaluations: 1616
Run 11: Best soluti